In [ ]:
#| export
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

In [ ]:
#| export
DEF_TZ = 'America/Chicago'

def today(tz=DEF_TZ):
    return datetime.now(ZoneInfo(tz)).date()

def add_days(n:int,date=None,tz=DEF_TZ):
    date = date or today(tz)
    return date+timedelta(days=n)

def make_day_offset(n):
    def offset(date=None, tz=DEF_TZ):
        return add_days(n, date, tz)
    return offset

ytd = make_day_offset(-1)
wk_ago = make_day_offset(-7)


In [ ]:
y=ytd()
print(y, type(y))

2026-01-15 <class 'datetime.date'>


In [ ]:
#| export
def str_to_date(content, year=None):
    if year is None:
        year = datetime.now().year

    formats = [
        (content,'%Y/%m/%d'),
        (content,'%Y-%m-%d'),
        (content,'%m/%d/%y'),
        (content,'%m/%d/%Y'),
        (f'{year}/{content}','%Y/%m/%d'),
        (f'{content} {year}', '%d %b %Y'),
        (f'{content} {year}', '%d %B %Y'),
        (f'{content} {year}', '%b %d %Y'),
        (f'{content} {year}', '%B %d %Y')
    ]

    for fmt in formats:
        try:
            return datetime.strptime(fmt[0], fmt[1]).date()
        except ValueError:
            continue
    raise ValueError(f"Could not parse date: {content}")
    

In [ ]:
str_to_date('2026-01-12')

datetime.date(2026, 1, 12)

Hi Solveit, A scheduled job (github actions) just failed with this error: 

Do my eyes deceive me or is there a wrong hyphen in the traceback date?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Your eyes do not deceive you! Look closely:

- Expected: `2026-01-12` (hyphen-minus `-`)
- In error: `2026–01-12` (en-dash `–` followed by hyphen-minus)

That first "hyphen" is actually an en-dash (Unicode U+2013), not a regular hyphen-minus (U+002D). This commonly happens when dates get copy-pasted from Word, Google Docs, or websites that auto-format hyphens into typographic dashes.

The fix would be in the source data, or you could make `str_to_date` more robust by normalizing dashes before parsing.

I'll fix the source. But I'm curious if there's a smarter way to write str_to_date treating any non-numeral as a separator?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes! You could use a regex to extract just the numeric parts, then work with those directly. Something like:

```python
import re
parts = re.split(r'\D+', '2026–01-12')
# parts = ['2026', '01', '12']
```

`\D+` matches one or more non-digit characters, so any separator (dash, en-dash, slash, space, etc.) gets treated the same way.

From there you could either reconstruct a normalized string, or build the date directly from the parts. Want to try implementing it?

Cases are:
- a 3-part date (which could come in several orders, 2025 12 31 or 12 31 25 or ...)
- a 2-part date with year passed in 
- a 2-part date without year passed in (use current year)
Wait: There are so many common ways to write dates, isn't there a standard library (or well-known non-standard library) that can just figure it out the way I do when I see a date?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes! `python-dateutil` has a `parser.parse()` that's great at guessing formats:

```python
from dateutil import parser
parser.parse('2026–01-12').date()  # handles the en-dash too!
```

It's a very popular library and handles a wide variety of formats automatically. It's already installed in solveit if you want to try it out.

There's also `dateparser` which is even more flexible (handles things like "yesterday" or "2 weeks ago"), but `dateutil` covers most standard date formats and is lighter weight.

In [ ]:
import re
def str_to_date(content,year=None)
    parts=re.split(r'\D+',content)
    

does the dateutil library provide the rest of the functionality in this module?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Looking at your module, `dateutil` covers:

- **Date parsing**: `parser.parse()` replaces `str_to_date` ✓
- **Date arithmetic**: `relativedelta` is more powerful than `timedelta` (e.g., "add 1 month") ✓

But your `today(tz)`, `add_days()`, and the offset helpers are simple enough that stdlib (`datetime`, `timedelta`, `ZoneInfo`) handles them fine. No real benefit swapping those to dateutil.

So you could simplify to just using `dateutil.parser.parse()` for `str_to_date` and keep the rest as-is.

I'm wondering if I can eliminate this module and just import dateutil where needed in the rest of the project. (You can read the entire project in this directory '.')

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠



<details class='tool-usage-details'>
<summary>view(path=.)</summary>

```json
{
  "id": "toolu_015TZBuyW1GTqU2hvdHNRkMS",
  "call": {
    "function": "view",
    "arguments": {
      "path": "."
    }
  },
  "result": "'Directory contents of /app/<TRUNCATED>"
}
```

</details>



<details class='tool-usage-details'>
<summary>rg(argstr=-l date_utils --type py .)</summary>

```json
{
  "id": "toolu_012EJQQfP27eEgFEJERMvSdK",
  "call": {
    "function": "rg",
    "arguments": {
      "argstr": "-l date_utils --type py ."
    }
  },
  "result": "'./qa_assist.py\\n./logs.py\\n'"
}
```

</details>



<details class='tool-usage-details'>
<summary>rg(argstr=from date_utils|import date_utils --type py .)</summary>

```json
{
  "id": "toolu_01TGC3v7eFe8avQNMFRciqa9",
  "call": {
    "function": "rg",
    "arguments": {
      "argstr": "from date_utils|import date_utils --type py ."
    }
  },
  "result": "'./weekly.py:2:from qa_assist import update_qdocs\\n./date_utils.py:1:from datetime import datetime, timedelta\\n./date_utils.py:2:from zoneinfo import ZoneInfo\\n./daily.py:2:from qa_assist import update_logs\\n./qdocs.py:1:from google_auth import get_gsheets_service\\n./emails.py:1:from datetime import datetime\\n./emails.py:2:from zoneinfo import ZoneInfo\\n./emails.py:3:from google_auth import get_gmail_service\\n./emails.py:19:    \"\"\"Get plain text body, timestamp, threadId, and sender from a Gmail message.\"\"\"\\n./emails.py:35:    \"\"\"Extract new content from email, removing quoted replies.\"\"\"\\n./l<TRUNCATED>"
}
```

</details>

Let me search more precisely:

<details class='tool-usage-details'>
<summary>rg(argstr=date_utils --type py .)</summary>

```json
{
  "id": "toolu_01HEJVTfr3jYafP1vBW69bLe",
  "call": {
    "function": "rg",
    "arguments": {
      "argstr": "date_utils --type py ."
    }
  },
  "result": "'./qa_assist.py:2:from date_utils import ytd, wk_ago\\n./logs.py:2:from date_utils import str_to_date\\n'"
}
```

</details>

So `date_utils` is used in two places:

1. **`logs.py`** imports `str_to_date`
2. **`qa_assist.py`** imports `ytd, wk_ago`

For `str_to_date`, you can replace it with `dateutil.parser.parse(...).date()`.

For `ytd` and `wk_ago` — those are convenience helpers for "yesterday" and "a week ago". You could either:
- Inline them where used (simple `datetime` arithmetic)
- Keep a small utility, or
- Use `dateutil.relativedelta`

Want me to show how those two files are using the functions, so you can decide if inlining makes sense?

str_to_date is used uniquely in logs.py, and ytd/wk_ago are used exclusively in qa_assist? 